# Session 2: propagating uncertainty with a formula
**A. Real-world problem.** A pendulum of length L and period T gives g = 4 pi^2 L / T^2.
You measure L with a ruler and T with a stopwatch. Both have uncertainty. What is the
uncertainty of g, and which measurement should you improve first?

**Plan.** (1) the standard error of the mean, (2) the propagation formula,
(3) check it by simulation, (4) error budget, (5) correlated inputs, (6) reporting.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: standard uncertainty and standard error of the mean
Repeat a reading n times. The standard deviation s of the readings is the *standard
uncertainty* of one reading. The mean of n readings is steadier: its uncertainty is the
*standard error of the mean*, s / sqrt(n). We verify this by simulating many experiments.

In [ ]:
def sem(s, n):
    """Standard error of the mean of n readings with standard deviation s."""
    return s / np.sqrt(n)


def simulate_sem(sigma, n, trials=20000, seed=1):
    """Spread of the means of `trials` experiments, each with n readings of spread sigma."""
    rng = np.random.default_rng(seed)
    means = rng.normal(10.0, sigma, size=(trials, n)).mean(axis=1)
    return float(means.std(ddof=1))

## Step 2: the propagation formula
For q = f(x, y) with small errors sx and sy:
    sq^2 = (df/dx)^2 sx^2 + (df/dy)^2 sy^2  + 2 (df/dx)(df/dy) cov(x, y)
The covariance term is zero for independent inputs. `linear_sigma` takes the list of
partial derivatives (the sensitivities) and the standard uncertainties; an optional
correlation rho (two inputs) adds the covariance cov = rho sx sy.

In [ ]:
def linear_sigma(grads, sigmas, rho=0.0):
    g = np.asarray(grads, dtype=float)
    s = np.asarray(sigmas, dtype=float)
    var = float(np.sum((g * s) ** 2))
    if len(g) == 2:
        var += 2.0 * g[0] * g[1] * rho * s[0] * s[1]
    return float(np.sqrt(max(var, 0.0)))


def variance_shares(grads, sigmas):
    """Fraction of the output variance that comes from each input (independent inputs)."""
    parts = (np.asarray(grads, dtype=float) * np.asarray(sigmas, dtype=float)) ** 2
    return parts / parts.sum()


def mc_sigma(f, means, sigmas, n=200000, seed=2, rho=0.0):
    """Monte Carlo spread of f for normal inputs (rho links the first two inputs)."""
    rng = np.random.default_rng(seed)
    k = len(means)
    z = rng.standard_normal((k, n))
    if k >= 2 and rho != 0.0:
        z[1] = rho * z[0] + np.sqrt(1.0 - rho ** 2) * z[1]
    xs = [m + s * zi for m, s, zi in zip(means, sigmas, z)]
    return float(np.std(f(*xs), ddof=1))

## Step 3: three formulas
Each function comes with its analytic partial derivatives, so we can compare the formula
with Monte Carlo.

In [ ]:
def area(L, W):
    return L * W


def area_grads(L, W):
    return [W, L]


def density(m, r, h):
    """Density of a cylinder: mass / (pi r^2 h)."""
    return m / (np.pi * r ** 2 * h)


def density_grads(m, r, h):
    rho = density(m, r, h)
    return [rho / m, -2.0 * rho / r, -rho / h]


def pendulum_g(L, T):
    """g = 4 pi^2 L / T^2 (small swings, point mass)."""
    return 4.0 * np.pi ** 2 * L / T ** 2


def pendulum_grads(L, T):
    g = pendulum_g(L, T)
    return [g / L, -2.0 * g / T]

## Step 4: reporting and the error budget
Quote the uncertainty with ONE significant figure, then round the value to the same
decimal place: g = 9.801 +- 0.109 is written 9.8 +- 0.1 m/s^2.

In [ ]:
def round_report(value, unc):
    """Return (value, unc, decimals): unc to one significant figure, value to the same place."""
    e = int(np.floor(np.log10(unc)))
    u1 = round(unc / 10.0 ** e) * 10.0 ** e
    if u1 >= 10.0 ** (e + 1):
        e += 1
    decimals = max(-e, 0)
    return round(value, -e), round(u1, -e), decimals


def pendulum_budget(L, sL, T, sT):
    """g, its uncertainty, the relative uncertainty and the variance share of L and T."""
    g = pendulum_g(L, T)
    gr = pendulum_grads(L, T)
    s = linear_sigma(gr, [sL, sT])
    return g, s, s / g, variance_shares(gr, [sL, sT])

## Step 5: see it
Figure 1: the pendulum Monte Carlo cloud of g with the linear Gaussian on top.
Figure 2: correlated inputs; the uncertainty of x - y falls as the correlation grows.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    L0, sL, T0, sT = 1.000, 0.005, 2.007, 0.010
    g0, sg, rel, sh = pendulum_budget(L0, sL, T0, sT)
    print(f"g = {g0:.3f}, linear sigma = {sg:.4f} ({100 * rel:.3f}%), shares L {sh[0]:.3f}, T {sh[1]:.3f}")
    print("Monte Carlo sigma =", round(mc_sigma(pendulum_g, [L0, T0], [sL, sT]), 4))
    print("report:", round_report(g0, sg))

    rng = np.random.default_rng(3)
    gs = pendulum_g(L0 + sL * rng.standard_normal(100000), T0 + sT * rng.standard_normal(100000))
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(gs, bins=80, density=True, alpha=0.5, label="Monte Carlo (100000)")
    xs = np.linspace(gs.min(), gs.max(), 300)
    ax.plot(xs, np.exp(-0.5 * ((xs - g0) / sg) ** 2) / (sg * np.sqrt(2 * np.pi)), "r", label="linear formula")
    ax.set_xlabel("g (m/s^2)")
    ax.set_ylabel("probability density (s^2/m)")
    ax.set_title("Pendulum: spread of g from L and T errors")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "pendulum_g.png", dpi=120)

    rhos = np.linspace(0, 1, 51)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(rhos, [linear_sigma([1, -1], [0.1, 0.1], r) for r in rhos], label="x - y")
    ax2.plot(rhos, [linear_sigma([1, 1], [0.1, 0.1], r) for r in rhos], label="x + y")
    ax2.set_xlabel("correlation rho between x and y (no unit)")
    ax2.set_ylabel("uncertainty of result (cm)")
    ax2.set_title("Shared calibration error cancels in a difference")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "correlation.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the exercises, answer the questions in the README (section F), run the checkpoint.

## Coding tasks
1. **Easy.** Use `simulate_sem(0.2, 5)` and `simulate_sem(0.2, 20)` and compare with `sem`; explain why quadrupling n halves the standard error.
2. **Stretch.** Add a third input to the cylinder density case: a temperature correction factor with 0.1 percent uncertainty, and update the variance shares.
3. **Explain.** Explain, with the correlation figure, why two lengths measured with the same stretched tape give a better difference than two different tapes.

## Assessment
- **A. Conceptual.** Why do we add uncertainties in quadrature (square, add, square root) instead of adding them directly? When would adding them directly be right?
- **B. Numerical.** A pendulum has L = 0.500 +- 0.002 m and T = 1.419 +- 0.005 s. Find g and its uncertainty, and say which input dominates.
- **C. Graph and simulation interpretation.** In the correlation figure, read the uncertainty of x - y and of x + y at rho = 0.9 for two inputs with u = 0.1 cm. What does a shared calibration error do?
- **D. Design and optimisation challenge.** Design: the target is a relative uncertainty of g of 0.6 percent or less. The stopwatch error for one swing is 0.1 s, so timing n swings and dividing gives u_T = 0.1 / n s. Choose n and the ruler error to meet the target. Is improving only the ruler enough?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # standard error of the mean
    checks.append(("SEM formula: s = 0.2, n = 5 gives 0.0894", abs(L.sem(0.2, 5) - 0.0894) < 1e-4))
    sim5, sim20 = L.simulate_sem(0.2, 5), L.simulate_sem(0.2, 20)
    checks.append(("simulation agrees with s/sqrt(n) within 3 percent (n = 5 and n = 20)",
                   abs(sim5 / L.sem(0.2, 5) - 1) < 0.03 and abs(sim20 / L.sem(0.2, 20) - 1) < 0.03))
    checks.append(("quadrupling n halves the standard error", abs(L.sem(0.2, 20) / L.sem(0.2, 5) - 0.5) < 1e-12))
    # worked example: pendulum
    g, s, rel, sh = L.pendulum_budget(1.000, 0.005, 2.007, 0.010)
    checks.append(("worked example: g = 9.801 m/s^2 and sigma_g = 0.109 m/s^2", abs(g - 9.801) < 5e-4 and abs(s - 0.109) < 5e-4))
    checks.append(("worked example: relative uncertainty of g is 1.1 percent", abs(100 * rel - 1.115) < 5e-3))
    checks.append(("worked example: variance share of L is 20 percent and of T is 80 percent",
                   abs(sh[0] - 0.20) < 0.005 and abs(sh[1] - 0.80) < 0.005))
    hand = 9.801 * np.sqrt(0.005 ** 2 + (2 * 0.010 / 2.007) ** 2)
    checks.append(("hand calculation sqrt((dL/L)^2 + (2 dT/T)^2) * g equals the code", abs(hand - s) < 5e-4))
    checks.append(("reporting rule: 9.801 +- 0.109 is written 9.8 +- 0.1", L.round_report(9.801, 0.109)[:2] == (9.8, 0.1)))
    checks.append(("reporting rule: 1234.5 +- 96 is written 1200 +- 100", L.round_report(1234.5, 96)[:2] == (1200, 100)))
    # linear formula against Monte Carlo
    mc = L.mc_sigma(L.pendulum_g, [1.0, 2.007], [0.005, 0.010])
    checks.append(("pendulum: Monte Carlo sigma agrees with the formula within 1 percent", abs(mc / s - 1) < 0.01))
    a_lin = L.linear_sigma(L.area_grads(12.3, 8.4), [0.1, 0.1])
    a_mc = L.mc_sigma(L.area, [12.3, 8.4], [0.1, 0.1])
    checks.append(("rectangle: formula 1.49 cm^2 agrees with Monte Carlo within 1 percent", abs(a_lin - 1.489) < 1e-3 and abs(a_mc / a_lin - 1) < 0.01))
    m, r, h, sg = 17.10, 0.635, 5.0, [0.01, 0.0005, 0.01]
    d_lin = L.linear_sigma(L.density_grads(m, r, h), sg)
    d_mc = L.mc_sigma(L.density, [m, r, h], sg)
    checks.append(("cylinder: density 2.700 g/cm^3, sigma 0.0071, Monte Carlo within 1 percent",
                   abs(L.density(m, r, h) - 2.700) < 5e-4 and abs(d_lin - 0.0071) < 5e-5 and abs(d_mc / d_lin - 1) < 0.01))
    shd = L.variance_shares(L.density_grads(m, r, h), sg)
    checks.append(("cylinder error budget: shares of mass, radius, height are 5, 36, 59 percent",
                   np.allclose(shd, [0.050, 0.364, 0.586], atol=0.002) and abs(shd.sum() - 1) < 1e-12))
    # special cases
    rng = np.random.default_rng(5)
    x = rng.normal(20, 0.5, 10 ** 6)
    y = rng.normal(30, 0.9, 10 ** 6)
    checks.append(("sums: errors add in quadrature (sqrt(0.5^2 + 0.9^2) = 1.03)",
                   abs((x + y).std() / np.hypot(0.5, 0.9) - 1) < 0.01))
    checks.append(("products: relative errors add in quadrature (2.5 % and 3 % give 3.9 %)",
                   abs((x * y).std() / 600 / np.hypot(0.5 / 20, 0.9 / 30) - 1) < 0.01 and abs(100 * np.hypot(0.025, 0.03) - 3.9) < 0.05))
    z = rng.normal(10, 0.2, 10 ** 6)
    checks.append(("powers: z^3 has 3 times the relative error of z (6 percent for 2 percent)",
                   abs((z ** 3).std() / (z ** 3).mean() / 0.06 - 1) < 0.02))
    # correlation
    d0 = L.linear_sigma([1, -1], [0.1, 0.1], 0.0)
    d9 = L.linear_sigma([1, -1], [0.1, 0.1], 0.9)
    d1 = L.linear_sigma([1, -1], [0.1, 0.1], 1.0)
    checks.append(("difference of two lengths: 0.141 cm if independent, 0.045 if rho = 0.9, 0 if rho = 1",
                   abs(d0 - 0.1414) < 5e-5 and abs(d9 - 0.0447) < 5e-5 and d1 < 1e-9))
    checks.append(("sum of two lengths grows with rho: 0.195 cm at rho = 0.9, 0.2 at rho = 1",
                   abs(L.linear_sigma([1, 1], [0.1, 0.1], 0.9) - 0.1949) < 5e-5 and abs(L.linear_sigma([1, 1], [0.1, 0.1], 1.0) - 0.2) < 1e-12))
    mc9 = L.mc_sigma(lambda a, b: a - b, [50.0, 30.0], [0.1, 0.1], rho=0.9)
    checks.append(("correlated Monte Carlo agrees: x - y with rho = 0.9 gives 0.045 cm", abs(mc9 / d9 - 1) < 0.02))
    # design problem
    b = {k: L.pendulum_budget(1.0, sl, 2.007, st)[2] * 100 for k, (sl, st) in
         {"now": (0.005, 0.010), "L": (0.0025, 0.010), "T": (0.005, 0.005), "both": (0.0025, 0.005), "perfectL": (0.0, 0.010)}.items()}
    checks.append(("design: halving the error of L gives 1.03 %, halving T gives 0.71 %, halving both gives 0.56 %",
                   abs(b["L"] - 1.027) < 5e-3 and abs(b["T"] - 0.706) < 5e-3 and abs(b["both"] - 0.557) < 5e-3))
    checks.append(("design: even a perfect length leaves 1.00 % because T dominates; only 'both' meets 0.6 %",
                   abs(b["perfectL"] - 0.997) < 5e-3 and b["both"] < 0.6 <= min(b["now"], b["L"], b["T"])))
    gB, sB, rB, shB = L.pendulum_budget(0.500, 0.002, 1.419, 0.005)
    checks.append(("question B: g = 9.803, u_g = 0.079, 0.81 percent, shares 24 and 76 percent, reported 9.80 +- 0.08",
                   abs(gB - 9.803) < 5e-4 and abs(sB - 0.0794) < 5e-4 and abs(100 * rB - 0.81) < 5e-3
                   and abs(shB[0] - 0.244) < 0.002 and L.round_report(gB, sB)[:2] == (9.8, 0.08)))
    checks.append(("question D: u_T = 0.1 / n gives 0.010 s for n = 10 and 0.005 s for n = 20; 20 swings alone give 0.71 %",
                   abs(0.1 / 10 - 0.010) < 1e-12 and abs(0.1 / 20 - 0.005) < 1e-12 and abs(b["T"] - 0.706) < 5e-3))
    checks.append(("difference u at rho = 0.9 is 0.045, sum 0.195 (question C), independent sum of two 0.1 errors is 0.141",
                   abs(d9 - 0.0447) < 5e-5 and abs(L.linear_sigma([1, 1], [0.1, 0.1]) - 0.1414) < 5e-5))
    checks.append(("edge: a zero-error input gives it a zero variance share", L.variance_shares([2.0, 3.0], [0.0, 0.1])[0] == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
